# Data Quality as a Finding

### Imports and preprocessing

In [1]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

# helper that saves each graph + summary so src/make_pdf.py can build the PDF
sys.path.insert(0, str(ROOT_DIR / "src"))
from report import save_report

event_countriesdf = pd.read_csv(Tables_DIR / "event_countries.csv")
eventsdf          = pd.read_csv(Tables_DIR / "events.csv")
country_yearlydf  = pd.read_csv(Tables_DIR / "country_yearly.csv")
global_yearlydf   = pd.read_csv(Tables_DIR / "global_yearly.csv")
coveragedf        = pd.read_csv(Tables_DIR / "data_coverage.csv")

for name, d in [("country_yearly", country_yearlydf), ("global_yearly", global_yearlydf),
                ("events", eventsdf), ("event_countries", event_countriesdf)]:
    print(name, list(d.columns))

# ---------- column auto-detection (edit candidates if needed) ----------
def col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"None of {names} found in {list(df.columns)}")

C_COUNTRY = col(country_yearlydf, "country", "country_name", "entity")
C_ISO     = col(country_yearlydf, "iso_code", "iso3", "country_code", "iso")
C_YEAR    = col(country_yearlydf, "year")
C_CO2     = col(country_yearlydf, "co2", "co2_mt", "annual_co2")
C_PC      = col(country_yearlydf, "co2_per_capita", "co2_pc", "co2_percap")
C_PM      = col(country_yearlydf, "pm25", "pm2_5", "pm25_exposure")

G_YEAR = col(global_yearlydf, "year")
G_TEMP = col(global_yearlydf, "temp_anomaly", "temperature_anomaly", "temp_anom", "anomaly", "j_d", "temp")

E_ID   = col(eventsdf, "event_id", "id")
E_YEAR = col(eventsdf, "year", "event_year")
E_NAME = col(eventsdf, "event_name", "name", "title", "event")
EC_ID  = col(event_countriesdf, "event_id", "id")
EC_ISO = col(event_countriesdf, "iso_code", "iso3", "country_code", "iso")

# real countries only (drop OWID aggregates like World, Asia, EU...)
is_country = (country_yearlydf[C_ISO].notna()
              & ~country_yearlydf[C_ISO].astype(str).str.startswith("OWID"))
cdf = country_yearlydf[is_country].copy()

# ---------- chart style ----------
NAVY, BLUE, RED, DARK = "#2c3e5e", "#4a78a8", "#e05252", "#2b2b2b"
FILL, GRID = "rgba(74,120,168,0.10)", "#e6eaf2"

def style(fig, title, subtitle, xt, yt, h=450, ylim=None):
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><sup>{subtitle}</sup>",
                   x=0.02, xanchor="left", font=dict(size=20, color=NAVY)),
        template="plotly_white", width=1090, height=h,
        margin=dict(l=75, r=45, t=95, b=95),
        font=dict(color=NAVY, size=13),
        plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.25),
    )
    fig.update_xaxes(title=xt, gridcolor=GRID, zeroline=False)
    fig.update_yaxes(title=yt, gridcolor=GRID, zeroline=False, range=ylim)
    return fig

def dot(x, y, label, color, name, pos="top left", showlegend=True):
    return go.Scatter(x=[x], y=[y], mode="markers+text", text=[label],
                      textposition=pos, name=name, showlegend=showlegend,
                      marker=dict(size=11, color=color), textfont=dict(color=NAVY))

def world_series(value_col):
    """World total per year = sum over real countries."""
    return cdf.groupby(C_YEAR)[value_col].sum(min_count=1)

country_yearly ['iso_code', 'country', 'year', 'population', 'gdp', 'co2_mt', 'co2_per_capita', 'cumulative_co2', 'share_global_co2', 'coal_co2', 'oil_co2', 'gas_co2', 'cement_co2', 'flaring_co2', 'methane', 'nitrous_oxide', 'total_ghg', 'pm25_exposure', 'co2_yoy_pct', 'rank_total_co2', 'rank_per_capita', 'co2_per_gdp', 'gdp_change_10yr_pct', 'co2_change_10yr_pct', 'decoupled_flag']
global_yearly ['year', 'decade', 'is_complete_year', 'co2_ppm', 'co2_mlo_ppm', 'ch4_ppb', 'n2o_ppb', 'temp_anomaly', 'world_co2_mt', 'co2_mlo_yoy_change', 'co2_pct_index', 'ch4_pct_index', 'n2o_pct_index', 'temp_yoy_change', 'co2_mlo_detrended', 'temp_detrended']
events ['event_id', 'event_name', 'year', 'event_type', 'source_note']
event_countries ['event_id', 'iso_code', 'role']


In [2]:
TOPIC = "data_quality"
aggdf = pd.read_csv(Tables_DIR / "aggregates_yearly.csv")

### Q1. Which variables have the most missing data and in which years/countries?

In [3]:
num = country_yearlydf.select_dtypes("number").drop(columns=[C_YEAR], errors="ignore")
miss = (num.isna().mean() * 100).sort_values(ascending=False).head(12)

# ---- graph 1: variables ----
fig = go.Figure(go.Bar(x=miss.values[::-1], y=miss.index[::-1], orientation="h", marker_color=BLUE,
                       text=[f"{v:.0f}%" for v in miss.values[::-1]], textposition="outside",
                       name="% missing"))
style(fig, "Variables With the Most Missing Data", "Share of country-year rows that are empty",
      "Missing (%)", "", h=500)
fig.update_xaxes(range=[0, 110])
fig.show()

# ---- graph 2: decades ----
dec = (country_yearlydf.assign(decade=(country_yearlydf[C_YEAR] // 10) * 10)
       .groupby("decade")[list(miss.index[:8])].apply(lambda g: g.isna().mean() * 100))
fig2 = go.Figure(go.Heatmap(z=dec.T.values, x=dec.index.astype(str) + "s", y=dec.columns,
                            colorscale=[[0, "#eef2f8"], [1, BLUE]], zmin=0, zmax=100,
                            colorbar=dict(title="% missing")))
style(fig2, "Missing Data by Decade", "Top 8 variables with gaps", "Decade", "", h=420)
fig2.show()

# ---- summary ----
worst = miss.index[0]
cnt = cdf.groupby(C_COUNTRY)[worst].apply(lambda s: s.isna().all()).sum()
summary = f"""
The most incomplete variable is "{worst}" ({miss.iloc[0]:.0f}% missing), followed by "{miss.index[1]}" ({miss.iloc[1]:.0f}%)
and "{miss.index[2]}" ({miss.iloc[2]:.0f}%). Gaps concentrate in the earliest decades, and {cnt} countries have no
"{worst}" values at all. Any analysis using these variables is effectively limited to later years and a subset of countries.
""".strip()
print(summary)
save_report(TOPIC, "q1", "Which variables have the most missing data and in which years/countries?",
            [fig, fig2], summary)

The most incomplete variable is "pm25_exposure" (84% missing), followed by "flaring_co2" (71%)
and "gdp_change_10yr_pct" (68%). Gaps concentrate in the earliest decades, and 23 countries have no
"pm25_exposure" values at all. Any analysis using these variables is effectively limited to later years and a subset of countries.
[report] saved data_quality/q1 (2 graph(s))


### Q2. Do the country sums match the world totals?

In [4]:
def is_text(s):
    return not pd.api.types.is_numeric_dtype(s)

def get_world_series():
    # shape A: a row labelled World/Global in any text column
    for d_ in (aggdf, country_yearlydf):
        ycol = next((c for c in d_.columns if c.lower() == "year"), None)
        co2c = next((c for c in d_.columns if c.lower() in ("co2", "co2_mt", "annual_co2")), None)
        if ycol is None or co2c is None:
            continue
        for c in [c for c in d_.columns if is_text(d_[c])]:
            hit = d_[c].astype(str).str.lower().str.strip().isin(["world", "global", "earth", "owid_wrl"])
            if hit.any():
                return d_[hit].set_index(ycol)[co2c]
    # shape B: a column like world_co2 / global_co2 in aggregates_yearly
    ycol = next((c for c in aggdf.columns if c.lower() == "year"), None)
    for c in aggdf.columns:
        if "co2" in c.lower() and ("world" in c.lower() or "global" in c.lower()):
            return aggdf.set_index(ycol)[c]
    return None

world_raw = get_world_series()
if world_raw is None:
    raise ValueError("No world CO2 total found in aggregates_yearly / country_yearly.")

world_raw = world_raw[~world_raw.index.duplicated()].dropna()
sums_raw = world_series(C_CO2).dropna()
print(f"World max = {world_raw.max():,.1f} | Sum of countries max = {sums_raw.max():,.1f}")

world_s, sums = world_raw / 1000, sums_raw / 1000       # Mt -> Gt
cmp_ = pd.concat([world_s.rename("world"), sums.rename("sum")], axis=1).dropna()
print(f"Overlapping years: {len(cmp_)}")
if cmp_.empty:
    raise ValueError("No overlapping years between world total and country sums.")

cmp_["gap"] = (cmp_["sum"] / cmp_["world"] - 1) * 100
worst_y = cmp_["gap"].abs().idxmax()

# ---- graph ----
fig = go.Figure()
fig.add_trace(go.Scatter(x=cmp_.index, y=cmp_["world"], mode="lines", name="World total",
                         line=dict(color=BLUE, width=3), fill="tozeroy", fillcolor=FILL))
fig.add_trace(go.Scatter(x=cmp_.index, y=cmp_["sum"], mode="lines", name="Sum of countries",
                         line=dict(color=RED, width=2, dash="dash")))
fig.add_trace(dot(worst_y, cmp_.loc[worst_y, "world"],
                  f"Largest gap<br>{worst_y}: {cmp_.loc[worst_y, 'gap']:+.1f}%",
                  DARK, "Largest gap", pos="bottom right"))
style(fig, "Country Sums vs World CO2 Total",
      f"Mean absolute gap {cmp_['gap'].abs().mean():.1f}%", "Year", "CO2 (Gt)",
      ylim=[0, cmp_[["world", "sum"]].max().max() * 1.1])
fig.show()

# ---- summary ----
summary = f"""
Summing countries gives, on average, a {cmp_['gap'].abs().mean():.1f}% difference from the reported world total, with the
largest gap in {worst_y} ({cmp_.loc[worst_y, 'gap']:+.1f}%). In {int((cmp_['gap'].abs() < 2).sum())} of {len(cmp_)} years the
two agree within 2%. Differences typically come from international aviation and shipping, territories without
ISO codes, and missing country-years.
""".strip()
print(summary)
save_report(TOPIC, "q2", "Do the country sums match the world totals?", fig, summary)

World max = 38,598.6 | Sum of countries max = 37,398.1
Overlapping years: 275


Summing countries gives, on average, a 0.8% difference from the reported world total, with the
largest gap in 2018 (-3.4%). In 205 of 275 years the
two agree within 2%. Differences typically come from international aviation and shipping, territories without
ISO codes, and missing country-years.
[report] saved data_quality/q2 (1 graph(s))


### Q3. How does the choice of baseline period change the temperature story?

In [5]:
tt = global_yearlydf.set_index(G_YEAR)[G_TEMP].dropna()
bases = {"1880-1900 (pre-industrial)": (1880, 1900), "1951-1980": (1951, 1980), "1991-2020": (1991, 2020)}
colors = [BLUE, "#8fb0d4", RED]

# ---- graph ----
fig = go.Figure()
vals = {}
for (name, (a, b_)), c in zip(bases.items(), colors):
    series = tt - tt.loc[a:b_].mean()
    vals[name] = series.iloc[-1]
    fig.add_trace(go.Scatter(x=series.index, y=series.values, mode="lines", name=name,
                             line=dict(color=c, width=2.5)))
fig.add_hline(y=0, line_color=GRID)
style(fig, "Same Data, Different Baselines",
      f"Latest anomaly ({tt.index[-1]}) ranges from {min(vals.values()):.2f} to {max(vals.values()):.2f} °C",
      "Year", "Temperature anomaly (°C)")
fig.show()

# ---- summary ----
vals_txt = ", ".join(f"{v:.2f} °C vs {k}" for k, v in vals.items())
summary = f"""
Rebasing the same temperature series changes the headline number: the {tt.index[-1]} anomaly is
{vals_txt}.
The shape and the warming trend are identical; only the zero line moves. A recent baseline makes warming look smaller and
a pre-industrial baseline makes it look larger, so the baseline must always be stated when quoting an anomaly.
""".strip()
print(summary)
save_report(TOPIC, "q3", "How does the choice of baseline period change the temperature story?", fig, summary)

Rebasing the same temperature series changes the headline number: the 2025 anomaly is
1.42 °C vs 1880-1900 (pre-industrial), 1.19 °C vs 1951-1980, 0.58 °C vs 1991-2020.
The shape and the warming trend are identical; only the zero line moves. A recent baseline makes warming look smaller and
a pre-industrial baseline makes it look larger, so the baseline must always be stated when quoting an anomaly.
[report] saved data_quality/q3 (1 graph(s))
